# Further Feature Combination Evaluation (SMOTE dataset)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA_PATH = "/content/drive/My Drive/WVS_happiness_study"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
X_train = pd.read_csv(f"{DRIVE_DATA_PATH}/data/train_sets/X_train_final_smote.csv")
X_test = pd.read_csv(f"{DRIVE_DATA_PATH}/data/test_sets/X_test_final.csv")
X_val = pd.read_csv(f"{DRIVE_DATA_PATH}/data/val_sets/X_val_final.csv")

y_train = pd.read_csv(f"{DRIVE_DATA_PATH}/data/train_sets/y_train_final_smote.csv")
y_test = pd.read_csv(f"{DRIVE_DATA_PATH}/data/test_sets/y_test_final.csv")
y_val = pd.read_csv(f"{DRIVE_DATA_PATH}/data/val_sets/y_val_final.csv")

## 5 Combinations of Features

In [3]:
feature_sets = {

    "set_1": [
        "sact",
        "physio1",
        "safety5",
        "belonging1",
        "esteem3",
        "trust3",
        "ethical80",
        "civic53",
        "compassion1",
        "compassion2",
        "compassion3",
        "compassion4",
        "growth16",
        "age",
        "marital_status",
        "religious1",
        "income_scale"
    ],

    "set_2": [
        "sact",
        "physio1",
        "safety5",
        "belonging1",
        "esteem3",
        "trust3",
        "ethical80",
        "civic53",
        "compassion2",
        "compassion3",
        "growth16",
        "age",
        "marital_status"
    ],

    "set_3": [ # maslow needs only
        "sact",
        "physio1",
        "safety5",
        "belonging1",
        "esteem3"
    ],

    "set_4": [ # maturity of heart only
        "trust3",
        "ethical80",
        "civic53",
        "compassion1",
        "compassion2",
        "growth16"
    ],

    "set_5": [ # hybrid
        "sact",
        "physio1",
        "safety5",
        "trust3",
        "ethical80"
    ],

    "set_6": [ # top 3
        "sact",
        "physio1",
        "safety5"
    ]
}

Create data splits.

In [4]:
data_splits = {}

for name, features in feature_sets.items():
    data_splits[name] = {
        "X_train": X_train[features],
        "X_val": X_val[features],
        "X_test": X_test[features],
        "feature_columns": features
    }

In [5]:
import warnings
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings("ignore")
pd.pandas.set_option("display.max_columns", None)

In [6]:
models = [
    {
        "name": "RandomForest",
        "estimator": RandomForestClassifier(random_state=42)
    },
    {
        "name": "LightGBM",
        "estimator": LGBMClassifier(random_state=42),
        "param_grid": {
            "n_estimators": [50, 100, 150],
            "learning_rate": [0.01, 0.05, 0.1],
            "max_depth": [-1, 5, 8]
        }
    },
    {
        "name": "LogisticRegression",
        "estimator": LogisticRegression(random_state=42, max_iter=3000)
    },
    {
        "name": "NaiveBayes",
        "estimator": GaussianNB()
    }
]


## Model Training

In [7]:
import sys
sys.path.append(f"{DRIVE_DATA_PATH}/code/my_ml_tools")
import training

all_results = {}
all_models = {}

for combo_name, data in data_splits.items():
    print(f"\n==============================")
    print(f"Running feature set: {combo_name}")
    print(f"==============================")

    results_df, best_models =  training.train_ensemble_models(
        data["X_train"], y_train,
        data["X_val"], y_val,
        data["X_test"], y_test,
        models,
        sort_by= "Test Accuracy"
    )

    all_results[combo_name] = results_df
    all_models[combo_name] = best_models


Running feature set: set_1
y_train: Converted to 1D pd automatically...
y_test: Converted to 1D pd automatically...
y_val: Converted to 1D pd automatically...

Training RandomForest...

Training LightGBM...
[LightGBM] [Info] Number of positive: 56869, number of negative: 56869
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.044839 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4335
[LightGBM] [Info] Number of data points in the train set: 113738, number of used features: 17
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive

In [8]:
from sklearn.metrics import classification_report

OUTPUT_PATH = f"{DRIVE_DATA_PATH}/data/feature_sets_performance/model_feature_comparison_smote.txt"

with open(OUTPUT_PATH, "w") as f:

    for combo_name, df in all_results.items():

        print(f"Evaluating: {combo_name}")

        f.write("\n" + "="*70 + "\n")
        f.write(f"FEATURE SET: {combo_name}, Predictors: {len(feature_sets[combo_name])}\n")
        f.write("="*70 + "\n\n")

        # Metrics table
        f.write(df.to_string(index=False))
        f.write("\n\n")
        f.write("Classification Reports:\n")

        # 🔥 IMPORTANT: use EXACT training feature columns
        X_test_subset = data_splits[combo_name]["X_test"]

        for model_name, model_info in all_models[combo_name].items():

            model = model_info["model"]

            # 🔥 HARD FIX: align columns EXACTLY to training time
            try:
                # safest direct prediction
                y_pred = model.predict(X_test_subset)

            except Exception:
                # fallback: force alignment to model's expected features (if available)
                if hasattr(model, "feature_names_in_"):
                    X_aligned = X_test_subset.reindex(
                        columns=model.feature_names_in_,
                        fill_value=0
                    )
                    y_pred = model.predict(X_aligned)
                else:
                    raise

            f.write(f"\n\n--- {model_name} ---\n")
            f.write(classification_report(y_test, y_pred, digits=4))

        f.write("\n\n")

print("INFO: Evaluation completed successfully!")

Evaluating: set_1
Evaluating: set_2
Evaluating: set_3
Evaluating: set_4
Evaluating: set_5
Evaluating: set_6
INFO: Evaluation completed successfully!


In [9]:
import joblib

logistic_models_per_combo = {}

for combo_name in all_results.keys():

    # search for LogisticRegression in trained models
    if "LogisticRegression" not in all_models[combo_name]:
        print(f"{combo_name}: LogisticRegression not found, skipping...")
        continue

    # get trained logistic regression model
    log_model = all_models[combo_name]["LogisticRegression"]["model"]

    # store in dict (optional)
    logistic_models_per_combo[combo_name] = log_model

    # save model
    filename = f"{DRIVE_DATA_PATH}/model/smote_logistic/smote_logistic_model_{combo_name}.pkl"
    joblib.dump(log_model, filename)

    print(f"{combo_name}: Saved LogisticRegression → {filename}")

set_1: Saved LogisticRegression → /content/drive/My Drive/WVS_happiness_study/model/smote_logistic/smote_logistic_model_set_1.pkl
set_2: Saved LogisticRegression → /content/drive/My Drive/WVS_happiness_study/model/smote_logistic/smote_logistic_model_set_2.pkl
set_3: Saved LogisticRegression → /content/drive/My Drive/WVS_happiness_study/model/smote_logistic/smote_logistic_model_set_3.pkl
set_4: Saved LogisticRegression → /content/drive/My Drive/WVS_happiness_study/model/smote_logistic/smote_logistic_model_set_4.pkl
set_5: Saved LogisticRegression → /content/drive/My Drive/WVS_happiness_study/model/smote_logistic/smote_logistic_model_set_5.pkl
set_6: Saved LogisticRegression → /content/drive/My Drive/WVS_happiness_study/model/smote_logistic/smote_logistic_model_set_6.pkl
